# Heatwave data analysis

## Things to analyze:
- Distance covered by mode of transport (per trip) vs temperature (actual movements, to see if the done movements are shorter/longer at different temps)
- Number of movements by mode of transport vs temperature (normalized by number of hour ranges at that temperature, to see if people move less at some temperatures)
- Number of steps vs max temperature of the day
- Number of steps vs mean temperature of the day
- Number of steps vs time, with temp vs time...should probably be a local analysis otherwise it's complicated.

## Imports

In [1]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import HeatMap
from branca.colormap import linear
from geolib import geohash as geolib
import json
import math
import random
import arc_drawer
import numpy as np
import pydeck as pdk
import seaborn as sns
import pickle as pkl
import requests
import matplotlib.pyplot as plt
import time

/Users/moreno/Documents/Lavoro/SWICE/Software/Data_analysis/SWICE_analysis/osmnx/lib/python3.9/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Define parameters

In [4]:
selected_mots = ['WALKING']  # Mode of transport to analyze
max_speed = 15  # Maximum speed in km/h for selected mode of transport (TODO: set different max speeds for different modes...!)

start_date = '2024-05-17'  # Start date for analysis -> None if no filtering
end_date = '2025-08-31'  # End date for analysis -> None if no filtering

bins_size = 2 # Size of temperature bins (in °C)

swiss_geohashes = [f"u0{c}" for c in "kmqrhjnp"] # Geohashes in switzerland

## Define functions

In [6]:
# Set age groups (over 60, 18-60, under 18)
def age_group(yob):
    if yob < 2025 - 60:
        return 'over 60'
    elif yob <= 2025 - 18:
        return '18-60'
    elif yob < 2025:
        return 'under 18'
    else:
        return 'unknown'

In [7]:
# Translate selected modes of transport to corresponding actual transport modes
def translate_mot(mot):
    if mot == "CAR" or mot == "ELECTRIC_CAR" or mot == "HYBRID_CAR":
        return "CAR"
    elif mot == "TRAIN":
        return "TRAIN"
    elif mot == "WALKING":
        return "WALKING"
    elif mot == "ON_BICYCLE" or mot == "ELECTRIC_BIKE" or mot == "SCOOTER" or mot == "ELECTRIC_SCOOTER":
        return "BICYCLE"
    elif mot == "BUS" or mot == "ELECTRIC_BUS" or mot == "COACH":
        return "BUS"
    elif mot == "TRAM":
        return "TRAM"
    elif mot == "PLANE":
        return "PLANE"
    elif mot == "BOAT" or mot == "BOAT_NO_ENGINE":
        return "BOAT"
    else:
        return mot

In [8]:
# Load caches if they exist
try:
    geohashes_cache = pkl.load(open("./streamlit/geohashes_to_coords.pkl", "rb"))
except FileNotFoundError:
    geohashes_cache = {}


try:
    temperature_cache = pkl.load(open("./data/temperature_cache.pkl", "rb"))
except FileNotFoundError:
    temperature_cache = {}


In [9]:
# Function to decode geohash with caching
def decode_geohash(geohash):
    if geohash in geohashes_cache:
        return geohashes_cache[geohash]
    lat, lon = geolib.decode(geohash)
    geohashes_cache[geohash] = (lat, lon)
    return (lat, lon)


In [10]:
# Function to get temperature from Open-Meteo API with caching and retries
def get_temp(geohash, dt):
    """Query Open-Meteo historical weather API for a specific time."""
    url = "https://archive-api.open-meteo.com/v1/archive"

    date = dt.strftime("%Y-%m-%d")

    # Check cache first
    if geohash+date in temperature_cache:
        data = temperature_cache[geohash+date]
    else:
        for _ in range(3):  # Retry up to 3 times
            try:
                lat, lon = decode_geohash(geohash)
                params = {
                    "latitude": lat,
                    "longitude": lon,
                    "start_date": date,
                    "end_date": date,
                    "hourly": "temperature_2m",
                    "timezone": "UTC"
                }
                r = requests.get(url, params=params)
                r.raise_for_status()
                data = r.json()
                # Cache the results
                if data:
                    temperature_cache[geohash+date] = data
                    break  # Exit loop if successful
                else:
                    print(f"No data returned for {geohash} on {date}")
                    return None
            except requests.RequestException as e:
                print(f"Error fetching data: {e}")
                # Save cache to file
                pkl.dump(temperature_cache, open("./data/temperature_cache.pkl", "wb"))
                time.sleep(2)  # Wait before retrying

    # Adjust hour to match the API's hourly data (which is on the hour)
    time_str = dt.strftime("%Y-%m-%dT%H:00")

    # Find the exact matching timestamp
    times = data["hourly"]["time"]
    temps = data["hourly"]["temperature_2m"]
    time_dict = dict(zip(times, temps))

    return time_dict.get(time_str, None)  # return temperature if found


In [11]:
# Compute mean and max temperatures for a trip
def compute_trip_temps(row):
    """Compute mean and max temperatures for a trip based on start and end geohashes and times."""
    # Get temps at start & end
    t1 = get_temp(row["start_geohash"], row["start_time_precise"])
    t2 = get_temp(row["end_geohash"], row["end_time_precise"])

    if t1 is None or t2 is None:
        return pd.Series({"mean_temp": None, "max_temp": None})

    return pd.Series({
        "mean_temp": (t1 + t2) / 2,
        "max_temp": max(t1, t2)
    })